In [22]:
!pip install icecream

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [icecream]


In [23]:
from icecream import ic

In [7]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# default: Load the model on the available device(s)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct", dtype="auto", device_map="auto"
)


Loading weights: 100%|██████████| 713/713 [00:02<00:00, 314.81it/s]


In [8]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# Preparation for inference
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)    

# Inference: Generation of the output
# generated_ids = model.generate(**inputs, max_new_tokens=128)
out = model.generate(
    **inputs,
    max_new_tokens=128,
    output_hidden_states=True,
    return_dict_in_generate=True,   # without this, output_hidden_states is ignored
)

generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, out.sequences)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)


print(output_text)



['This is a heartwarming, sun-drenched photograph capturing a tender moment between a woman and her dog on a beach at sunset.\n\n**Key Elements:**\n\n- **The Subjects:** A young woman with long dark hair, wearing a plaid shirt and dark pants, is sitting cross-legged in the sand. She is smiling warmly, looking at her dog. Beside her, a large, light-colored Labrador Retriever, wearing a colorful patterned harness, sits attentively, extending its front paw to give a high-five with the woman’s hand.\n\n- **The Setting:** The scene is set on a wide, sandy beach.']


In [10]:
out.keys()

odict_keys(['sequences', 'hidden_states', 'past_key_values'])

In [19]:
out['hidden_states']

((tensor([[[-0.0011,  0.0205,  0.0192,  ...,  0.0136,  0.0184, -0.0052],
           [ 0.0400,  0.0308,  0.0077,  ..., -0.0039,  0.0259,  0.0074],
           [-0.0527, -0.0136,  0.0525,  ..., -0.0315, -0.0293,  0.0183],
           ...,
           [-0.0011,  0.0205,  0.0192,  ...,  0.0136,  0.0184, -0.0052],
           [-0.0106,  0.0216,  0.0071,  ...,  0.0239,  0.0114, -0.0175],
           [-0.0527, -0.0136,  0.0525,  ..., -0.0315, -0.0293,  0.0183]]],
         device='cuda:0', dtype=torch.bfloat16),
  tensor([[[ 5.9688, -1.9531,  1.3047,  ...,  0.1719, -0.1572,  0.0552],
           [ 6.0000, -1.2812,  1.3359,  ...,  0.0737, -0.1260, -0.0269],
           [ 7.5312, -1.2188,  2.5938,  ..., -0.0127, -0.0938, -0.0256],
           ...,
           [ 1.6094, -1.3906,  0.8242,  ..., -0.0095, -0.0183,  0.0422],
           [ 2.9062, -0.9141,  0.8789,  ..., -0.0322, -0.0413, -0.1455],
           [ 3.9375, -0.4805,  1.6328,  ...,  0.0317, -0.0732,  0.0181]]],
         device='cuda:0', dtype=torch.b

In [24]:
ic(len(out['hidden_states']), len(out['hidden_states'][0]), out['hidden_states'][0][0].shape)

ic| len(out['hidden_states']): 128
    len(out['hidden_states'][0]): 37
    out['hidden_states'][0][0].shape: torch.Size([1, 2766, 2560])


(128, 37, torch.Size([1, 2766, 2560]))

In [34]:
layer_id = 12
out['hidden_states'][0][layer_id].shape

torch.Size([1, 2766, 2560])